In [0]:
# Check if sentence-transformers is already installed
try:
    import sentence_transformers
    print('✅ sentence-transformers already installed')
except ImportError:
    print('📦 Installing sentence-transformers...')
    %pip install -q sentence-transformers
    print('⚠️  Restarting Python (MLflow will not work after restart)')
    dbutils.library.restartPython()

# Loop de Convergencia - Clasificación Jerárquica Iterativa

## Objetivo
Procesar productos no clasificados en **batches pequeños (100 productos)** con **recálculo de centroids** después de cada batch exitoso.

## Estrategia
1. **Iterar N veces** (ej: 30 iteraciones = 3,000 productos)
2. **Por cada iteración:**
   - Tomar 100 productos random no clasificados
   - Clasificar con thresholds altos:
     - `MACRO_AUTO = 0.90`
     - `CATEGORY_AUTO = 0.75`
   - Si hay productos auto-asignados:
     - Guardar en `gold_productos_categorias`
     - **Ejecutar notebook 05** para recalcular centroids
     - Recargar centroids en memoria
   - Si NO hay productos auto-asignados:
     - Continuar (centroids no cambian)

3. **Tracking de convergencia:**
   - % de requeue por iteración
   - Productos auto-asignados acumulados
   - Ver si mejora con el tiempo

## Hipótesis
A medida que agregamos productos, los centroids se refinan y **deberían capturar mejor la distribución de productos "difíciles"** → el % de requeue debería **disminuir**.

---

In [0]:
import json
import numpy as np
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer
import time
import os

# Configure MLflow for serverless - set tracking URI via environment variable to avoid Spark config access
os.environ['MLFLOW_TRACKING_URI'] = 'databricks'
os.environ['MLFLOW_EXPERIMENT_NAME'] = '/Users/franciscobarber@gmail.com/superapp/classification_convergence'

import mlflow
mlflow.autolog(disable=True)  # prevent autologging hooks

print('✅ MLflow configured for serverless')

# Configuration - LOWERED THRESHOLDS for better classification rate
MACRO_AUTO = 0.70      # Threshold para auto-asignar MACRO (was 0.90, too high)
CATEGORY_AUTO = 0.60   # Threshold para auto-asignar CATEGORIA (was 0.75, too high)
BATCH_SIZE = 100       # Productos por iteracion
MAX_ITERATIONS = 5     # Start with 5 iterations for testing (was 30)
REBUILD_CENTROIDS_EVERY = 3  # Recalcular centroids cada N iteraciones
MIN_ACCURACY_PCT = 80.0      # Minimo accuracy para promover a gold

# LLM Configuration
USE_LLM = False  # endpoint not configured yet
LLM_MODEL = 'databricks-meta-llama-3-3-70b-instruct'
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
LLM_MACRO_MIN = 0.60   # Lowered from 0.75
LLM_MACRO_MAX = 0.70   # Adjusted to match MACRO_AUTO
LLM_TOP_CANDIDATES = 3

print(f'Configuracion:')
print(f'   MACRO_AUTO:              {MACRO_AUTO} (lowered for better classification)')
print(f'   CATEGORY_AUTO:           {CATEGORY_AUTO} (lowered for better classification)')
print(f'   BATCH_SIZE:              {BATCH_SIZE}')
print(f'   MAX_ITERATIONS:          {MAX_ITERATIONS} (testing mode)')
print(f'   REBUILD_CENTROIDS_EVERY: {REBUILD_CENTROIDS_EVERY}')
print(f'   MIN_ACCURACY_PCT:        {MIN_ACCURACY_PCT}%')
print(f'   USE_LLM:                 {USE_LLM}')
print(f'   Total productos a procesar: {BATCH_SIZE * MAX_ITERATIONS:,}')

In [0]:
def build_macro_prompt(producto, macro_names):
    """Construye prompt para que LLM decida el MACRO"""
    macros_text = ", ".join(macro_names)
    return f"""Eres un clasificador experto de productos de supermercado.

Producto: "{producto}"

Macros disponibles: {macros_text}

¿A qué macro pertenece este producto? Responde SOLO con JSON:
{{
    "macro": "nombre_del_macro",
    "confianza": 0.0-1.0
}}

Si no estás seguro, usa "ninguna" como macro."""

def build_category_prompt(producto, top_candidates, all_categories_dict):
    """Construye prompt para que LLM decida la CATEGORIA"""
    candidates_text = "\n".join([
        f"- {cat} (similitud={sim:.2f})" + 
        (f" ej: {examples[0] if isinstance(examples, list) and examples else examples}" if examples else "")
        for cat, sim, examples in top_candidates
    ])
    
    return f"""Eres un clasificador experto de productos de supermercado.

Producto: "{producto}"

Categorías candidatas:
{candidates_text}

¿A qué categoría pertenece este producto? Responde SOLO con JSON:
{{
    "categoria": "nombre_de_la_categoria",
    "confianza": 0.0-1.0
}}

Si no estás seguro, usa "ninguna" como categoría."""

print('✅ Funciones build_macro_prompt() y build_category_prompt() definidas')

In [0]:
# 🧪 EXPERIMENT ISOLATION FUNCTIONS
# Safe experimentation: staging → validate → promote

def create_experiment_staging_table(run_id):
    """
    Creates a staging table for this experiment run.
    Returns: staging_table_name
    """
    staging_table = f"workspace.superapp.experiment_classifications_{run_id[:8]}"
    
    print(f'📋 Creating staging table: {staging_table}')
    
    # Create table with Delta Lake property to allow column defaults
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {staging_table} (
            id_producto BIGINT,
            id_categoria BIGINT,
            id_subcategoria BIGINT,
            metodo STRING,
            confianza DOUBLE,
            fecha_asignacion TIMESTAMP,
            usuario_asignacion STRING,
            notas STRING,
            run_id STRING,
            iteration INT
        )
        USING DELTA
        TBLPROPERTIES (
            'delta.feature.allowColumnDefaults' = 'supported'
        )
    """)
    
    print(f'✅ Staging table created: {staging_table}')
    return staging_table

def save_to_staging(records, staging_table, run_id, iteration=None):
    """
    Saves classification records to staging table.
    Returns: number of records saved
    """
    if not records:
        return 0
    
    from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, TimestampType, IntegerType
    
    # Add run_id and iteration to each record
    for record in records:
        record['run_id'] = run_id
        record['iteration'] = iteration
        if 'id_subcategoria' not in record:
            record['id_subcategoria'] = None
    
    # Create pandas DataFrame and explicitly convert types
    pdf = pd.DataFrame(records)
    pdf['id_producto'] = pdf['id_producto'].astype('int64')
    pdf['id_categoria'] = pdf['id_categoria'].astype('int64')
    pdf['id_subcategoria'] = pdf['id_subcategoria'].astype('Int64')
    pdf['confianza'] = pdf['confianza'].astype('float64')
    pdf['fecha_asignacion'] = pd.to_datetime(pdf['fecha_asignacion'])
    pdf['iteration'] = pdf['iteration'].astype('Int32') if iteration is not None else None
    
    # Define schema explicitly
    schema = StructType([
        StructField('id_producto', LongType(), False),
        StructField('id_categoria', LongType(), False),
        StructField('id_subcategoria', LongType(), True),
        StructField('metodo', StringType(), False),
        StructField('confianza', DoubleType(), False),
        StructField('fecha_asignacion', TimestampType(), False),
        StructField('usuario_asignacion', StringType(), False),
        StructField('notas', StringType(), True),
        StructField('run_id', StringType(), False),
        StructField('iteration', IntegerType(), True)
    ])
    
    df = spark.createDataFrame(pdf, schema=schema)
    df.write.mode('append').saveAsTable(staging_table)
    
    print(f'   💾 Saved {len(records)} records to staging')
    return len(records)

def validate_staging_table(staging_table):
    """
    Validates classifications in staging table.
    Returns: dict with validation metrics
    """
    print(f'\n🔍 Validating {staging_table}...')
    
    validation_df = spark.sql(f"""
        SELECT
            COUNT(*) as total_classifications,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%vino%' OR LOWER(sp.producto) LIKE '%wine%')
                     AND c.nombre != 'Vinos'
                THEN 1 ELSE 0 
            END), 0) as wine_errors,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%cerveza%' OR LOWER(sp.producto) LIKE '%beer%')
                     AND c.nombre != 'Cerveza'
                THEN 1 ELSE 0
            END), 0) as beer_errors,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%whisky%' OR LOWER(sp.producto) LIKE '%whiskey%')
                     AND c.nombre != 'Whisky'
                THEN 1 ELSE 0
            END), 0) as whisky_errors,
            COALESCE(SUM(CASE WHEN c.nombre LIKE '_OLD_%' THEN 1 ELSE 0 END), 0) as old_category_errors
        FROM {staging_table} st
        JOIN workspace.superapp.silver_prices sp ON st.id_producto = sp.id_producto
        JOIN workspace.superapp.gold_categorias c ON st.id_categoria = c.id_categoria
    """).collect()[0]
    
    total = validation_df['total_classifications']
    detectable_errors = (
        validation_df['wine_errors'] + 
        validation_df['beer_errors'] + 
        validation_df['whisky_errors'] + 
        validation_df['old_category_errors']
    )
    estimated_accuracy_pct = ((total - detectable_errors) / total * 100) if total > 0 else 0
    
    print(f'\n📊 VALIDATION RESULTS:')
    print(f'   Total classifications: {total:,}')
    print(f'   Detectable errors: {detectable_errors:,}')
    print(f'   Estimated accuracy: {estimated_accuracy_pct:.1f}%')
    
    return {
        'total_classifications': total,
        'detectable_errors': detectable_errors,
        'estimated_accuracy_pct': estimated_accuracy_pct
    }

def promote_to_gold(staging_table, min_accuracy_pct=80.0):
    """
    Promotes validated classifications to gold table if accuracy meets threshold.
    Returns: True if promoted, False if validation failed
    """
    validation = validate_staging_table(staging_table)
    
    if validation['estimated_accuracy_pct'] >= min_accuracy_pct:
        print(f'\n✅ VALIDATION PASSED ({validation["estimated_accuracy_pct"]:.1f}% >= {min_accuracy_pct}%)')
        print(f'   Promoting {validation["total_classifications"]:,} classifications to gold...')
        
        spark.sql(f"""
            INSERT INTO workspace.superapp.gold_productos_categorias
            SELECT 
                id_producto,
                id_categoria,
                id_subcategoria,
                metodo,
                confianza,
                fecha_asignacion,
                usuario_asignacion,
                notas
            FROM {staging_table}
        """)
        
        print(f'   ✅ Promoted to gold_productos_categorias')
        return validation["total_classifications"]
    else:
        print(f'\n❌ VALIDATION FAILED ({validation["estimated_accuracy_pct"]:.1f}% < {min_accuracy_pct}%)')
        return 0

print('✅ Experiment isolation functions defined')

In [0]:
def load_centroids():
    """
    Carga centroids de MACRO y CATEGORIA desde las tablas gold.
    Retorna: (macro_matrix, macro_names, macro_id_map, categories_by_macro)
    """
    # Load MACRO centroids (9 macros)
    macro_pd = spark.table('workspace.superapp.gold_macro_centroids').toPandas()
    macro_pd['centroid'] = macro_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    macro_matrix = np.array(macro_pd['centroid'].tolist())  # (9, 384)
    macro_names = macro_pd['nombre'].tolist()
    macro_id_map = dict(zip(macro_pd['nombre'], macro_pd['id_macro']))
    
    # Load CATEGORY centroids (139 categories) with macro mapping
    cat_pd = spark.sql("""
        SELECT 
            cc.id_categoria,
            cc.nombre,
            cc.centroid_json,
            c.id_macro
        FROM workspace.superapp.gold_category_centroids cc
        JOIN workspace.superapp.gold_categorias c ON cc.id_categoria = c.id_categoria
        WHERE c.id_macro IS NOT NULL
          AND cc.nombre NOT LIKE '\_OLD\_%'
          AND cc.nombre NOT LIKE '\_DELETED\_%'
    """).toPandas()
    
    cat_pd['centroid'] = cat_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    # Group categories by macro
    categories_by_macro = {}
    for macro_id in macro_id_map.values():
        macro_cats = cat_pd[cat_pd['id_macro'] == macro_id]
        if len(macro_cats) > 0:
            categories_by_macro[macro_id] = {
                'names': macro_cats['nombre'].tolist(),
                'ids': macro_cats['id_categoria'].tolist(),
                'matrix': np.array(macro_cats['centroid'].tolist())
            }
    
    return macro_matrix, macro_names, macro_id_map, categories_by_macro

print('✅ Función load_centroids() definida')

In [0]:
from datetime import datetime
import numpy as np

def cosine_sim_batch(embeddings, centroid_matrix):
    """Calcula similitud coseno entre embeddings y centroids"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / (norms + 1e-10)
    return normed @ centroid_matrix.T

def classify_batch_phase1(batch_df, model, macro_matrix, macro_names, macro_id_map, categories_by_macro, all_categories_dict, 
                          macro_auto_threshold, category_auto_threshold, llm_macro_min, llm_macro_max, use_llm=False):
    """
    FASE 1: Calcula embeddings, similitudes, y prepara preguntas para LLM batch.
    NO llama al LLM todavía - solo acumula las preguntas.
    
    Args:
        batch_df: DataFrame con productos a clasificar
        model: modelo de embeddings
        macro_matrix, macro_names, macro_id_map: datos de macros
        categories_by_macro: categorías organizadas por macro
        all_categories_dict: ejemplos de categorías para LLM
        macro_auto_threshold: threshold para auto-asignar por macro (adaptive)
        category_auto_threshold: threshold para auto-asignar por categoría (adaptive)
        llm_macro_min: threshold mínimo para usar LLM
        llm_macro_max: threshold máximo para usar LLM
        use_llm: si usar LLM o no
    
    Retorna: (auto_assigned, llm_questions, product_context)
    - auto_assigned: productos que se auto-asignan sin LLM (alta confianza)
    - llm_questions: lista de preguntas para procesar en batch con Genie
    - product_context: dict con contexto de cada producto para uso posterior
    """
    auto_assigned = []
    llm_questions = []  # [{id_producto, tipo, prompt, context}, ...]
    product_context = {}  # {id_producto: {macro_id, cat_matrix, cat_names, cat_ids, ...}}
    
    # Generar embeddings
    embs = model.encode(batch_df['producto'].tolist(), batch_size=128, show_progress_bar=False)
    
    # LEVEL 1: Classify by MACRO
    macro_sims = cosine_sim_batch(embs, macro_matrix)
    macro_top_idx = macro_sims.argmax(axis=1)
    macro_top_scores = macro_sims.max(axis=1)
    
    for i, (_, row) in enumerate(batch_df.iterrows()):
        macro_score = float(macro_top_scores[i])
        macro_idx = macro_top_idx[i]
        macro_name_embedding = macro_names[macro_idx]
        id_producto = row['id_producto']
        producto = row['producto']
        
        # ========================================
        # PATH 1: FULL LLM (macro_sim < llm_macro_min)
        # ========================================
        if use_llm and macro_score < llm_macro_min:
            # Preparar pregunta de MACRO para LLM
            prompt = build_macro_prompt(producto, macro_names)
            
            llm_questions.append({
                'id_producto': id_producto,
                'producto': producto,
                'tipo': 'macro',
                'prompt': prompt,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score
            })
            
            # Guardar embedding del producto para fase 2
            product_context[id_producto] = {
                'embedding': embs[i],
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score
            }
            
            continue
        
        # ========================================
        # PATH 2 & 3: Embedding decides MACRO
        # ========================================
        
        macro_id = macro_id_map[macro_name_embedding]
        
        # Check if macro has categories
        if macro_id not in categories_by_macro:
            # Skip - no categories in this macro
            continue
        
        macro_data = categories_by_macro[macro_id]
        cat_matrix = macro_data['matrix']
        cat_names = macro_data['names']
        cat_ids = macro_data['ids']
        
        cat_sims = cosine_sim_batch(embs[i:i+1], cat_matrix)[0]
        cat_top_idx = cat_sims.argmax()
        cat_top_score = float(cat_sims[cat_top_idx])
        cat_name = cat_names[cat_top_idx]
        cat_id = cat_ids[cat_top_idx]
        
        # ========================================
        # PATH 3: HIGH confidence (macro ≥ llm_macro_max, cat ≥ category_auto_threshold)
        # ========================================
        if macro_score >= llm_macro_max and cat_top_score >= category_auto_threshold:
            auto_assigned.append({
                'id_producto': id_producto,
                'id_categoria': int(cat_id),
                'id_subcategoria': None,
                'metodo': 'embedding_hierarchical_convergence',
                'confianza': cat_top_score,
                'fecha_asignacion': datetime.now(),
                'usuario_asignacion': 'system',
                'notas': f'macro={macro_name_embedding}({macro_score:.3f})|cat={cat_name}({cat_top_score:.3f})'
            })
            
        # ========================================
        # PATH 2: MEDIUM macro confidence (llm_macro_min ≤ macro < llm_macro_max) - LLM validates category
        # ========================================
        elif use_llm and llm_macro_min <= macro_score < llm_macro_max:
            # Get top N candidates with examples
            top_indices = cat_sims.argsort()[-3:][::-1]  # LLM_TOP_CANDIDATES = 3
            top_candidates = []
            for idx in top_indices:
                cat_candidate = cat_names[idx]
                sim = float(cat_sims[idx])
                examples = all_categories_dict.get(cat_candidate, [])
                top_candidates.append((cat_candidate, sim, examples))
            
            # Preparar pregunta de CATEGORIA para LLM
            prompt = build_category_prompt(producto, top_candidates, all_categories_dict)
            
            llm_questions.append({
                'id_producto': id_producto,
                'producto': producto,
                'tipo': 'categoria',
                'prompt': prompt,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score,
                'embedding_cat': cat_name,
                'embedding_cat_sim': cat_top_score
            })
            
            # Guardar contexto para fase 2
            product_context[id_producto] = {
                'macro_id': macro_id,
                'cat_names': cat_names,
                'cat_ids': cat_ids,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score,
                'embedding_cat': cat_name,
                'embedding_cat_sim': cat_top_score
            }
    
    return auto_assigned, llm_questions, product_context

print('✅ Función classify_batch_phase1() definida (acepta thresholds adaptativos)')

In [0]:
from datetime import datetime
import json
import pandas as pd

def process_llm_batch(llm_questions, product_context, macro_id_map, categories_by_macro, model):
    """
    FASE 2: Procesa todas las preguntas LLM en batch usando Genie (ai_query).
    
    Args:
        llm_questions: lista de preguntas preparadas en fase 1
        product_context: dict con contexto de cada producto
        macro_id_map: mapeo de nombre macro a ID
        categories_by_macro: dict con categorías por macro
        model: modelo de embeddings (para caso full_path)
    
    Returns:
        (auto_assigned, review_queue, stats)
    """
    if not llm_questions:
        return [], [], {'llm_called': 0, 'llm_success': 0, 'llm_full_path': 0, 'llm_category_only': 0}
    
    print(f'   🤖 Procesando {len(llm_questions)} preguntas LLM en batch con Genie...')
    
    # 1. Crear tabla temporal con preguntas
    questions_df = pd.DataFrame(llm_questions)
    spark_questions = spark.createDataFrame(questions_df)
    spark_questions.createOrReplaceTempView('llm_batch_questions')
    
    # 2. Ejecutar ai_query en batch
    results_df = spark.sql("""
        SELECT 
            id_producto,
            tipo,
            ai_query('superapp_endpoint', prompt) as respuesta,
            embedding_macro,
            embedding_macro_sim,
            embedding_cat,
            embedding_cat_sim
        FROM llm_batch_questions
    """).toPandas()
    
    print(f'   ✅ Genie procesó {len(results_df)} respuestas')
    
    # 3. Parse respuestas y clasificar
    auto_assigned = []
    review_queue = []
    stats = {'llm_called': len(llm_questions), 'llm_success': 0, 'llm_full_path': 0, 'llm_category_only': 0}
    
    for _, row in results_df.iterrows():
        id_producto = row['id_producto']
        tipo = row['tipo']
        respuesta_raw = row['respuesta']
        context = product_context.get(id_producto, {})
        
        # Parse JSON response
        try:
            respuesta = json.loads(respuesta_raw)
        except:
            # JSON parsing error - review queue
            review_queue.append({
                'id_producto': id_producto,
                'producto': llm_questions[0]['producto'],  # lookup from original
                'razon': 'llm_json_parse_error',
                'top_macro': row.get('embedding_macro'),
                'similitud_macro': row.get('embedding_macro_sim'),
                'top_categoria': row.get('embedding_cat'),
                'similitud': row.get('embedding_cat_sim'),
                'llm_sugerencia': respuesta_raw[:100],
                'estado': 'pendiente',
                'fecha_creacion': datetime.now()
            })
            continue
        
        # ========================================
        # TIPO: MACRO (full LLM path)
        # ========================================
        if tipo == 'macro':
            stats['llm_full_path'] += 1
            
            llm_macro = respuesta.get('macro', 'ninguna')
            llm_macro_conf = float(respuesta.get('confianza', 0.0))
            
            if llm_macro == 'ninguna' or llm_macro_conf < 0.7:
                # LLM uncertain - review queue
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_low_confidence',
                    'top_macro': row.get('embedding_macro'),
                    'similitud_macro': row.get('embedding_macro_sim'),
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': f'{llm_macro}({llm_macro_conf:.2f})',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # LLM confirmed macro - now decide category
            macro_id = macro_id_map.get(llm_macro)
            if not macro_id or macro_id not in categories_by_macro:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_not_found',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': llm_macro,
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # Use embedding to find best category within LLM-selected macro
            macro_data = categories_by_macro[macro_id]
            cat_matrix = macro_data['matrix']
            cat_names = macro_data['names']
            cat_ids = macro_data['ids']
            
            embedding = context['embedding']
            cat_sims = cosine_sim_batch(embedding.reshape(1, -1), cat_matrix)[0]
            cat_top_idx = cat_sims.argmax()
            cat_top_score = float(cat_sims[cat_top_idx])
            cat_name = cat_names[cat_top_idx]
            cat_id = cat_ids[cat_top_idx]
            
            if cat_top_score >= 0.60:  # Lower threshold for full LLM path
                auto_assigned.append({
                    'id_producto': id_producto,
                    'id_categoria': int(cat_id),
                    'id_subcategoria': None,
                    'metodo': 'embedding_hierarchical_llm_full_convergence',
                    'confianza': (llm_macro_conf + cat_top_score) / 2,  # Average
                    'fecha_asignacion': datetime.now(),
                    'usuario_asignacion': 'system',
                    'notas': f'llm_macro={llm_macro}({llm_macro_conf:.3f})|emb_cat={cat_name}({cat_top_score:.3f})'
                })
                stats['llm_success'] += 1
            else:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_ok_but_low_category',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': cat_name,
                    'similitud': cat_top_score,
                    'llm_sugerencia': f'{llm_macro}>{cat_name}',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
        
        # ========================================
        # TIPO: CATEGORIA (hybrid path)
        # ========================================
        elif tipo == 'categoria':
            stats['llm_category_only'] += 1
            
            llm_cat = respuesta.get('categoria', 'ninguna')
            llm_cat_conf = float(respuesta.get('confianza', 0.0))
            
            if llm_cat == 'ninguna' or llm_cat_conf < 0.7:
                # LLM uncertain - review queue
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_category_low_confidence',
                    'top_macro': context.get('embedding_macro'),
                    'similitud_macro': context.get('embedding_macro_sim'),
                    'top_categoria': context.get('embedding_cat'),
                    'similitud': context.get('embedding_cat_sim'),
                    'llm_sugerencia': f'{llm_cat}({llm_cat_conf:.2f})' if llm_cat != 'ninguna' else 'ninguna',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # LLM confirmed category - find its ID
            cat_names = context['cat_names']
            cat_ids = context['cat_ids']
            
            llm_cat_id = None
            for idx, name in enumerate(cat_names):
                if name == llm_cat:
                    llm_cat_id = cat_ids[idx]
                    break
            
            if llm_cat_id:
                auto_assigned.append({
                    'id_producto': id_producto,
                    'id_categoria': int(llm_cat_id),
                    'id_subcategoria': None,
                    'metodo': 'embedding_hierarchical_llm_convergence',
                    'confianza': llm_cat_conf,
                    'fecha_asignacion': datetime.now(),
                    'usuario_asignacion': 'system',
                    'notas': f'emb_macro={context.get("embedding_macro")}({context.get("embedding_macro_sim"):.3f})|llm_cat={llm_cat}({llm_cat_conf:.3f})'
                })
                stats['llm_success'] += 1
            else:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_category_not_found',
                    'top_macro': context.get('embedding_macro'),
                    'similitud_macro': context.get('embedding_macro_sim'),
                    'top_categoria': llm_cat,
                    'similitud': llm_cat_conf,
                    'llm_sugerencia': llm_cat,
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
    
    return auto_assigned, review_queue, stats

print('✅ Función process_llm_batch() definida (procesa preguntas con Genie en batch)')

In [0]:
def rebuild_centroids():
    """
    Recalcula centroids inline usando todos los productos clasificados.
    Retorna True si exitoso, False si falla.
    """
    try:
        print('   🔧 Recalculando centroids...')
        
        # 1. Load all classified products with their categories
        labeled = spark.sql("""
            SELECT DISTINCT
                pc.id_producto,
                COALESCE(pc.id_subcategoria, pc.id_categoria) AS id_categoria_final,
                COALESCE(gcs.nombre, gc.nombre) AS categoria_nombre,
                sp.producto
            FROM workspace.superapp.gold_productos_categorias pc
            LEFT JOIN workspace.superapp.gold_categorias gc
                ON pc.id_categoria = gc.id_categoria
            LEFT JOIN workspace.superapp.gold_categorias gcs
                ON pc.id_subcategoria = gcs.id_categoria
            JOIN workspace.superapp.silver_prices sp
                ON pc.id_producto = sp.id_producto
            WHERE pc.id_categoria IS NOT NULL
        """).toPandas()
        
        if len(labeled) == 0:
            print('   ⚠️  No hay productos clasificados aún')
            return False
            
        print(f'   📊 Productos clasificados: {len(labeled):,}')
        
        # 2. Generate embeddings (using same model as classification)
        embeddings = model.encode(
            labeled['producto'].tolist(),
            show_progress_bar=False,
            batch_size=128
        )
        labeled['embedding'] = list(embeddings)
        
        # 3. Compute centroids per category
        centroids = []
        for cat_nombre, group in labeled.groupby('categoria_nombre'):
            vecs = np.array(group['embedding'].tolist())
            centroid = vecs.mean(axis=0)
            norm = np.linalg.norm(centroid)
            if norm > 1e-10:
                centroid = centroid / norm
            
            centroids.append({
                'id_categoria': int(group['id_categoria_final'].iloc[0]),
                'nombre': cat_nombre,
                'n_productos': len(group),
                'centroid_json': json.dumps(centroid.tolist())
            })
        
        centroids_pd = pd.DataFrame(centroids).sort_values('n_productos', ascending=False)
        print(f'   📌 Centroids calculados: {len(centroids_pd)}')
        
        # 4. Save to table
        (spark.createDataFrame(centroids_pd[['id_categoria', 'nombre', 'n_productos', 'centroid_json']])
              .write.mode('overwrite')
              .option('overwriteSchema', 'true')
              .saveAsTable('workspace.superapp.gold_category_centroids'))
        
        print('   ✅ Centroids guardados exitosamente')
        return True
        
    except Exception as e:
        print(f'   ❌ Error al recalcular centroids: {str(e)[:150]}')
        import traceback
        traceback.print_exc()
        return False

print('✅ Función rebuild_centroids() definida (inline)')

In [0]:
# 🧪 LOOP WITH EXPERIMENT ISOLATION & STAGING TABLES
# Classifications go to staging first → validate → promote to gold if accurate

import uuid
from datetime import datetime
import mlflow
import mlflow.tracking._model_registry.utils

# Workaround for serverless: prevent MLflow from accessing spark.mlflow.modelRegistryUri
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

def _mlf_param(key, value):
    try: mlflow.log_param(key, value)
    except Exception: pass

def _mlf_metric(key, value, step=None):
    try:
        mlflow.log_metric(key, value, step=step) if step is not None else mlflow.log_metric(key, value)
    except Exception: pass

print('🔧 Iniciando MLflow tracking...')
_mlflow_active = False
try:
    _mlf_run = mlflow.start_run(run_name=f"classification_convergence_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    run_id = _mlf_run.info.run_id
    _mlflow_active = True
    print(f'✅ MLflow Run ID: {run_id}')
except Exception as _e:
    run_id = uuid.uuid4().hex[:8]
    print(f'⚠️  MLflow unavailable ({_e.__class__.__name__}) — continuing without tracking. run_id={run_id}')

try:
    # 🧪 CREATE EXPERIMENT STAGING TABLE
    print(f'\n🧪 EXPERIMENT ISOLATION ENABLED')
    staging_table = create_experiment_staging_table(run_id)
    _mlf_param("staging_table", staging_table)
    
    # Dynamic thresholds - will adjust based on performance
    current_macro_auto = MACRO_AUTO
    current_category_auto = CATEGORY_AUTO
    current_llm_min = LLM_MACRO_MIN
    current_llm_max = LLM_MACRO_MAX
    
    print(f'\n🎯 THRESHOLDS INICIALES (adaptativos):')
    print(f'   MACRO_AUTO:     {current_macro_auto:.2f} (se ajustará según performance)')
    print(f'   CATEGORY_AUTO:  {current_category_auto:.2f} (se ajustará según performance)')
    print(f'   LLM_MIN:        {current_llm_min:.2f}')
    print(f'   LLM_MAX:        {current_llm_max:.2f}')
    
    # Log parameters
    _mlf_param("macro_auto_initial", current_macro_auto)
    _mlf_param("category_auto_initial", current_category_auto)
    _mlf_param("batch_size", BATCH_SIZE)
    _mlf_param("max_iterations", MAX_ITERATIONS)
    _mlf_param("rebuild_centroids_every", REBUILD_CENTROIDS_EVERY)
    _mlf_param("use_llm", USE_LLM)
    _mlf_param("llm_model", LLM_MODEL)
    _mlf_param("embedding_model", EMBEDDING_MODEL)
    
    # Load embedding model
    print('\n🧠 Cargando modelo e5-small...')
    model = SentenceTransformer(EMBEDDING_MODEL)
    print('✅ Modelo e5-small cargado')
    
    # Load initial centroids
    print('\n🔧 Cargando centroids iniciales...')
    macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
    print(f'✅ Centroids cargados: {len(macro_names)} macros, {sum(len(v["names"]) for v in categories_by_macro.values())} categorías')
    
    # Load all categories with examples (for LLM)
    print('\n🔧 Cargando ejemplos de categorías para LLM...')
    cat_examples = spark.sql("""
        SELECT 
            c.nombre,
            get(COLLECT_LIST(sp.producto), 0) as producto
        FROM workspace.superapp.gold_categorias c
        LEFT JOIN workspace.superapp.gold_productos_categorias gpc ON c.id_categoria = gpc.id_categoria
        LEFT JOIN workspace.superapp.silver_prices sp ON gpc.id_producto = sp.id_producto
        WHERE c.nivel = 'categoria' AND c.is_active = TRUE
        GROUP BY c.nombre
    """)
    
    all_categories_dict = {}
    for _, row in cat_examples.toPandas().iterrows():
        if pd.notna(row['producto']):
            all_categories_dict[row['nombre']] = row['producto']
    
    print(f'✅ Cargados ejemplos de {len(all_categories_dict)} categorías')
    
    # Tracking variables
    iteration_results = []
    
    print('\n' + '='*80)
    print('🚀 INICIANDO LOOP DE CONVERGENCIA (CON STAGING TABLES)')
    print('='*80)
    
    for i in range(MAX_ITERATIONS):
        print(f'\n--- Iteración {i+1}/{MAX_ITERATIONS} ---')
        print(f'   Thresholds: MACRO={current_macro_auto:.2f}, CATEGORY={current_category_auto:.2f}, LLM_MIN={current_llm_min:.2f}, LLM_MAX={current_llm_max:.2f}')
        
        # Get unclassified products - EXCLUDE both gold AND staging
        unclassified = spark.sql(f"""
            SELECT sp.id_producto, sp.producto
            FROM workspace.superapp.silver_prices sp
            LEFT JOIN workspace.superapp.gold_productos_categorias gpc 
                ON sp.id_producto = gpc.id_producto
            LEFT JOIN {staging_table} stg
                ON sp.id_producto = stg.id_producto
            WHERE gpc.id_producto IS NULL
              AND stg.id_producto IS NULL
        """)
        
        unclassified_count = unclassified.count()
        print(f'Productos no clasificados: {unclassified_count:,}')
        
        if unclassified_count == 0:
            print('✅ No hay más productos para clasificar')
            break
        
        # Sample batch
        if unclassified_count < BATCH_SIZE:
            print(f'⚠️  Solo {unclassified_count} productos restantes')
            sample = unclassified.toPandas()
        else:
            sample = unclassified.sample(fraction=min(1.0, BATCH_SIZE/unclassified_count)).limit(BATCH_SIZE).toPandas()
        
        print(f'Batch size: {len(sample)}')
        
        # PHASE 1: Classify with embeddings (PASS ADAPTIVE THRESHOLDS)
        print(f'\n🔍 FASE 1: Clasificando con embeddings...')
        auto_assigned_phase1, llm_questions, product_context = classify_batch_phase1(
            sample, model, macro_matrix, macro_names, macro_id_map, categories_by_macro, all_categories_dict,
            macro_auto_threshold=current_llm_max,  # Use current_llm_max as high confidence threshold
            category_auto_threshold=current_category_auto,  # Use adaptive category threshold
            llm_macro_min=current_llm_min,
            llm_macro_max=current_llm_max,
            use_llm=USE_LLM
        )
        
        print(f'   ✅ Auto-asignados (embeddings): {len(auto_assigned_phase1)}')
        print(f'   🤖 Preguntas LLM generadas: {len(llm_questions)}')
        
        # PHASE 2: Process LLM batch
        auto_assigned_llm = []
        review_queue = []
        llm_stats = {'total': 0, 'success': 0, 'full_path': 0, 'category_only': 0}
        
        if USE_LLM and llm_questions:
            print(f'\n🤖 FASE 2: Ejecutando {len(llm_questions)} preguntas LLM INLINE...')
            auto_assigned_llm, review_queue, llm_stats = process_llm_batch(
                llm_questions, product_context, macro_id_map, categories_by_macro, model
            )
        
        # Combine auto-assigned
        auto_assigned = auto_assigned_phase1 + auto_assigned_llm
        
        # Results
        print(f'\n📊 RESULTADOS ITERACIÓN {i+1}:')
        print(f'   ✅ Total auto-asignados: {len(auto_assigned)} ({len(auto_assigned)/len(sample)*100:.1f}%)')
        print(f'      - Por embeddings: {len(auto_assigned_phase1)}')
        print(f'      - Por LLM: {len(auto_assigned_llm)}')
        print(f'   ⚠️  Review queue: {len(review_queue)} ({len(review_queue)/len(sample)*100:.1f}%)')
        
        # Save to STAGING
        saved_count = save_to_staging(auto_assigned, staging_table, run_id, iteration=i+1)
        
        # Log iteration metrics
        _mlf_metric("auto_assigned", len(auto_assigned), step=i)
        _mlf_metric("review_queue", len(review_queue), step=i)
        _mlf_metric("auto_rate", len(auto_assigned)/len(sample) if len(sample) > 0 else 0, step=i)
        
        # Track results
        iteration_results.append({
            'iteration': i+1,
            'batch_size': len(sample),
            'auto_assigned': len(auto_assigned),
            'auto_phase1': len(auto_assigned_phase1),
            'auto_phase2': len(auto_assigned_llm),
            'review_queue': len(review_queue),
            'macro_auto': current_macro_auto,
            'category_auto': current_category_auto,
            'llm_min': current_llm_min
        })
        
        # Save review queue
        if review_queue:
            review_pdf = pd.DataFrame(review_queue)
            review_df = spark.createDataFrame(review_pdf)
            review_df.write.mode('append').saveAsTable('workspace.superapp.gold_review_queue')
            print(f'   💾 Review queue guardada: {len(review_queue)} productos')
        
        # ADAPTIVE THRESHOLDS
        auto_rate = len(auto_assigned) / len(sample) if len(sample) > 0 else 0
        
        if auto_rate < 0.40:
            current_category_auto = max(0.50, current_category_auto - 0.05)
            print(f'   📉 Reduciendo CATEGORY_AUTO a {current_category_auto:.2f} (auto rate: {auto_rate:.1%})')
        elif auto_rate > 0.60:
            current_category_auto = min(0.80, current_category_auto + 0.05)
            print(f'   📈 Aumentando CATEGORY_AUTO a {current_category_auto:.2f} (auto rate: {auto_rate:.1%})')
        
        # Recalculate centroids every N iterations
        if len(auto_assigned) > 0 and (i + 1) % REBUILD_CENTROIDS_EVERY == 0:
            print(f'\n🔧 Recalculando centroids (iteración {i+1})...')
            success = rebuild_centroids()
            if success:
                macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
                print(f'✅ Centroids recargados')
    
    # ========================================
    # VALIDATION & PROMOTION
    # ========================================
    print('\n' + '='*80)
    print('✅ LOOP COMPLETADO - INICIANDO VALIDACIÓN')
    print('='*80)
    
    total_auto = sum(r['auto_assigned'] for r in iteration_results)
    print(f'\n🔍 Validando {total_auto} clasificaciones en staging...')
    
    if total_auto > 0:
        validation = validate_staging_table(staging_table)
        accuracy = validation["estimated_accuracy_pct"]
        error_count = validation["detectable_errors"]
        validation_passed = accuracy >= MIN_ACCURACY_PCT
        
        if validation_passed:
            promoted_count = promote_to_gold(staging_table, min_accuracy_pct=MIN_ACCURACY_PCT)
            print(f'\n✅ EXPERIMENT SUCCESSFUL - Classifications promoted to gold')
            print(f'   Staging table can be dropped: {staging_table}')
            
            _mlf_metric("validation_accuracy", accuracy)
            _mlf_metric("promoted_count", promoted_count)
        else:
            print(f'\n❌ VALIDATION FAILED - Accuracy {accuracy:.1%} < 80%')
            print(f'   Keeping staging table for review: {staging_table}')
            
            _mlf_metric("validation_accuracy", accuracy)
            _mlf_metric("validation_failed", 1)
    else:
        print('⚠️  No classifications to validate (0 products auto-assigned)')
        _mlf_metric("validation_accuracy", 0)
    
    # Final metrics
    total_processed = sum(r['batch_size'] for r in iteration_results)
    
    _mlf_metric("total_processed", total_processed)
    _mlf_metric("total_auto_assigned", total_auto)
    _mlf_metric("final_category_auto", current_category_auto)
    
    # Summary
    print('\n' + '='*80)
    print('📊 RESUMEN FINAL')
    print('='*80)
    print(f'Experiment Run ID:         {run_id}')
    print(f'Staging Table:             {staging_table}')
    print(f'Total iteraciones:         {len(iteration_results)}')
    print(f'Total productos procesados: {total_processed}')
    print(f'Total auto-asignados:      {total_auto} ({total_auto/total_processed*100:.1f}% if total_processed > 0 else 0)')
    print(f'\n🎯 Evolución de Thresholds:')
    print(f'   CATEGORY_AUTO:  {CATEGORY_AUTO:.2f} → {current_category_auto:.2f} (delta: {current_category_auto-CATEGORY_AUTO:+.2f})')
    print('='*80)
    
except Exception as e:
    print(f'\n❌ Error en loop: {e}')
    import traceback
    traceback.print_exc()
    _mlf_param('error', str(e)[:200])
    raise
finally:
    if _mlflow_active and mlflow.active_run():
        mlflow.end_run()

print('\n✅ Loop completado')

In [0]:
# Start with EXACT working pattern from ML notebook
import mlflow

print('🧪 Testing minimal MLflow pattern...')

with mlflow.start_run():
    # Just log basic params/metrics like working notebook
    mlflow.log_param("test_param", "value1")
    mlflow.log_metric("test_metric", 0.99)
    
    print('✅ MLflow basic logging works!')
    print(f'   Run ID: {mlflow.active_run().info.run_id}')

In [0]:
# Create DataFrame with results
import pandas as pd
import matplotlib.pyplot as plt

results_df = pd.DataFrame(iteration_results)

print('
📊 RESULTADOS DETALLADOS POR ITERACION')
print('='*100)
print(results_df[[
    'iteration', 'batch_size', 'auto_assigned', 'auto_phase1', 'auto_phase2',
    'review_queue', 'macro_auto', 'category_auto'
]].to_string(index=False))

print('
🔍 ANALISIS')
print('='*100)

if results_df['auto_assigned'].sum() > 0:
    total_auto = results_df['auto_assigned'].sum()
    total_phase1 = results_df['auto_phase1'].sum()
    total_phase2 = results_df['auto_phase2'].sum()
    total_review = results_df['review_queue'].sum()
    total_processed = results_df['batch_size'].sum()

    print(f'✅ Total auto-asignados: {total_auto}')
    print(f'   - Por embeddings (alta confianza): {total_phase1} ({total_phase1/total_auto*100:.1f}%)')
    print(f'   - Por LLM batch:                  {total_phase2} ({total_phase2/total_auto*100:.1f}%)')
    print(f'
📋 Review queue: {total_review:,} ({total_review/total_processed*100:.1f}% del total)')

    if len(results_df) > 1:
        print(f'
🎯 Evolucion de thresholds:')
        print(f'   MACRO_AUTO:    {results_df["macro_auto"].iloc[0]:.2f} -> {results_df["macro_auto"].iloc[-1]:.2f}')
        print(f'   CATEGORY_AUTO: {results_df["category_auto"].iloc[0]:.2f} -> {results_df["category_auto"].iloc[-1]:.2f}')
else:
    print('❌ NO hubo productos auto-asignados')
    print('   -> Los thresholds son muy altos o los embeddings no capturan bien estos productos')

print('='*100)


In [0]:
# Mostrar ejemplos de preguntas LLM y respuestas
if 'llm_batch_questions' in dir():
    print('\n🔍 EJEMPLOS DE PREGUNTAS LLM (Batch Processing)')
    print('='*100)
    
    # Leer las preguntas que quedaron en la tabla temporal
    try:
        questions_sample = spark.sql("""
            SELECT 
                id_producto,
                producto,
                tipo,
                LEFT(prompt, 200) as prompt_preview,
                embedding_macro,
                embedding_macro_sim
            FROM llm_batch_questions
            LIMIT 5
        """).toPandas()
        
        print(f'\nMostrando {len(questions_sample)} ejemplos de preguntas:')
        print('-'*100)
        
        for i, row in questions_sample.iterrows():
            print(f"\n{i+1}. Producto: {row['producto']}")
            print(f"   Tipo: {row['tipo']}")
            print(f"   Embedding macro: {row['embedding_macro']} (sim={row['embedding_macro_sim']:.3f})")
            print(f"   Prompt preview: {row['prompt_preview']}...")
            print('-'*100)
    
    except Exception as e:
        print(f'No se pudieron leer ejemplos: {e}')

print('\n✅ Análisis completo - revisar resultados arriba')